## imports

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

# Paths to your cleaned data
files = {
    "Benin": Path("data/benin_clean.csv"),
    "Sierra Leone": Path("data/sierraleon_clean.csv"),
    "Togo": Path("data/togo_clean.csv")
}

dfs = []
for country, f in files.items():
    if f.exists():
        df = pd.read_csv(f, parse_dates=["Timestamp"], dayfirst=False)
        df["Country"] = country
        dfs.append(df)
    else:
        print(f"⚠ Missing: {f}")

all_df = pd.concat(dfs, ignore_index=True)
metrics = [c for c in ["GHI","DNI","DHI"] if c in all_df.columns]
all_df.head()


## boxplots per metric

In [ ]:
for m in metrics:
    plt.figure(figsize=(7,5))
    data = [all_df.loc[all_df["Country"]==c, m].dropna() for c in all_df["Country"].unique()]
    plt.boxplot(data, labels=all_df["Country"].unique())
    plt.title(f"{m} by Country")
    plt.ylabel(m)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## summary table

In [ ]:
summary = (
    all_df.groupby("Country")[metrics]
          .agg(["mean","median","std","count"])
          .round(2)
)
display(summary)


## ANOVA and Kruskal-Wallis tests

In [ ]:
if "GHI" in metrics:
    groups = [all_df.loc[all_df["Country"]==c, "GHI"].dropna() for c in all_df["Country"].unique()]
    if all(len(g) > 2 for g in groups):
        anova = stats.f_oneway(*groups)
        kruskal = stats.kruskal(*groups)
        print(f"ANOVA on GHI:  F={anova.statistic:.3f},  p={anova.pvalue:.4g}")
        print(f"Kruskal–Wallis on GHI:  H={kruskal.statistic:.3f},  p={kruskal.pvalue:.4g}")
    else:
        print("Not enough data per group for tests.")


## ranking by average GHI

In [ ]:
if "GHI" in metrics:
    rank = all_df.groupby("Country")["GHI"].mean().sort_values(ascending=False)
    plt.figure(figsize=(6,4))
    rank.plot(kind="bar", color="skyblue", edgecolor="black")
    plt.title("Average GHI (higher is better)")
    plt.ylabel("Mean GHI (W/m²)")
    plt.tight_layout()
    plt.show()
    display(rank.round(2))


## summary observations (markdown or text)

In [ ]:
from IPython.display import Markdown as md

observations = """
### Key Observations (Week 0 Task 3)
- **Solar potential:** Country with the highest average GHI shows the strongest solar resource.
- **Variability:** Boxplot spread reveals day-to-day stability differences—narrower boxes mean more consistent sunlight.
- **Statistical significance:** ANOVA/Kruskal p-values below 0.05 indicate meaningful differences in GHI across countries.
- **Recommendation:** Prioritize countries with high mean GHI and low variance for solar farm investment feasibility.
"""
md(observations)
